# NumPy：用“形状、轴、广播”理解数组计算

**核心心智模型**

- `shape`：每个维度有多长；
- `dtype`：元素如何存储；
- `axis`：沿哪个维度做操作；
- 广播：不同形状如何自动对齐；
- 向量化：一次描述整批数据的计算。

机器学习里常用约定：`X.shape == (样本数, 特征数)`。


## 初学者使用 NumPy 的心智模型

如果用 C++ 类比：

- Python `list` 类似灵活的动态容器；
- NumPy `ndarray` 更像一块**类型统一、连续布局、带多维 shape 信息**的数值内存；
- NumPy 运算会一次作用于整块数组，而不是让 Python 逐元素循环。

阅读任何 NumPy 代码时，先写下：

1. 每个数组的 `shape`；
2. 每个数组的 `dtype`；
3. 操作沿哪个 `axis`；
4. 操作是逐元素、聚合，还是矩阵乘法；
5. 结果是否为 view，修改后会不会影响原数组。

常见 API 形式：

- `np.sum(arr, axis=0)`：函数式调用；
- `arr.sum(axis=0)`：对象方法调用；
- 两者在这里表达相同操作。

> NumPy 报错时不要先改代码，先打印所有参与运算对象的 `.shape`。


In [1]:
import numpy as np

# 控制数组打印格式：保留 3 位小数，并避免用科学计数法显示小数。
np.set_printoptions(precision=3, suppress=True)
# default_rng 创建独立随机数生成器；固定种子 42 可复现实验。
rng = np.random.default_rng(42)
print("NumPy version:", np.__version__)


NumPy version: 2.4.3


## 1. 创建数组与查看属性

`ndim` 是维度数，`shape` 是各维长度，`size` 是元素总数，`dtype` 是存储类型。


In [2]:
vector = np.array([1, 2, 3], dtype=np.float64)
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])

for name, arr in {"vector": vector, "matrix": matrix}.items():
    print(f"{name}: shape={arr.shape}, ndim={arr.ndim}, "
          f"size={arr.size}, dtype={arr.dtype}")


vector: shape=(3,), ndim=1, size=3, dtype=float64
matrix: shape=(2, 3), ndim=2, size=6, dtype=int64


In [3]:
# zeros/ones 的参数是 shape；dtype 控制元素存储类型。
zeros = np.zeros((2, 3))
ones = np.ones((2, 3), dtype=int)
# arange 使用 [start, stop) 和步长；linspace 指定两端及元素个数。
sequence = np.arange(0, 10, 2)     # [start, stop)，步长为 2
grid = np.linspace(0, 1, 5)        # 包含两端，共 5 个点
random_values = rng.normal(loc=0, scale=1, size=(2, 3))

print("arange:", sequence)
print("linspace:", grid)
print("random:\n", random_values)


arange: [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
random:
 [[ 0.305 -1.04   0.75 ]
 [ 0.941 -1.951 -1.302]]


### `dtype` 与类型转换

`dtype` 决定每个元素如何存储，也影响内存和计算结果。`astype()` 返回转换后的新数组。

- 整数除法通常产生浮点结果；
- 浮点数转整数会直接截断小数部分；
- `np.empty` 只分配内存，不会自动清零，初始值不可依赖。


In [4]:
integers = np.array([1, 2, 3], dtype=np.int32)
# astype 返回类型转换后的新数组，不会原地修改 integers。
floats = integers.astype(np.float64)
truncated = np.array([1.9, -2.7]).astype(np.int32)

print(integers.dtype, floats.dtype)
print("浮点转整数会截断:", truncated)

uninitialized = np.empty((2, 3))
print("empty 的值不可依赖，只能确认 shape:", uninitialized.shape)


int32 float64
浮点转整数会截断: [ 1 -2]
empty 的值不可依赖，只能确认 shape: (2, 3)


## 2. 索引、切片与布尔筛选

二维数组用 `arr[行, 列]`。切片通常返回**视图**，修改视图可能影响原数组；需要独立数据时调用 `.copy()`。


In [5]:
arr = np.arange(12).reshape(3, 4)
print("原数组:\n", arr)
print("第 2 行:", arr[1])
print("前 2 行、第 2~3 列:\n", arr[:2, 1:3])
print("大于 5 的元素:", arr[arr > 5])

# 普通切片通常是共享内存的 view；copy() 创建独立数据。
view = arr[:, :2]
independent = arr[:, :2].copy()
view[0, 0] = -99
print("修改 view 后 arr[0, 0]:", arr[0, 0])
print("copy 仍保持:", independent[0, 0])


原数组:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
第 2 行: [4 5 6 7]
前 2 行、第 2~3 列:
 [[1 2]
 [5 6]]
大于 5 的元素: [ 6  7  8  9 10 11]
修改 view 后 arr[0, 0]: -99
copy 仍保持: 0


### 按行迭代与按元素迭代

直接迭代二维数组时，每次得到一行；`.flat` 提供按元素的一维迭代器。性能敏感的计算仍应优先使用向量化，而不是 Python 循环。


In [6]:
matrix = np.arange(6).reshape(2, 3)
print("逐行:")
for row in matrix:
    print(row)

print("逐元素:", list(matrix.flat))


逐行:
[0 1 2]
[3 4 5]
逐元素: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 3. 形状变换

`reshape` 改变观察方式但不改变元素总数；`T` 交换二维数组的行列；`ravel` 展平。


In [7]:
values = np.arange(12)
matrix = values.reshape(3, 4)

print("matrix shape:", matrix.shape)
print("transpose shape:", matrix.T.shape)
print("flattened:", matrix.ravel())

column = np.array([10, 20, 30]).reshape(-1, 1)  # -1 让 NumPy 自动推断
print("column shape:", column.shape)


matrix shape: (3, 4)
transpose shape: (4, 3)
flattened: [ 0  1  2  3  4  5  6  7  8  9 10 11]
column shape: (3, 1)


### 转置、展平与 `np.newaxis`

- `.T` / `np.transpose` 交换轴；
- `.ravel()` 尽量返回视图（如果原数组的内存布局允许，.ravel() 不会复制数据，而是返回一个“看起来是一维”的视图；如果没法直接视图化，才会复制一份新数组。）；
- `.flatten()` 总是返回副本；
- `np.newaxis` 等价于 `None`，用于增加长度为 1 的维度。

一维数组 `(n,)` 转置后仍是 `(n,)`。要得到列向量，应写 `arr[:, np.newaxis]`。


In [8]:
matrix = np.arange(12).reshape(3, 4)
vector = np.array([1, 2, 3])

# transpose 交换轴；ravel 尽量返回 view；flatten 总是复制。
print("transpose:", np.transpose(matrix).shape)
print("ravel:", matrix.ravel().shape)
print("flatten:", matrix.flatten().shape)
print("一维转置:", vector.T.shape)
print("列向量:", vector[:, np.newaxis].shape)
print("行向量:", vector[np.newaxis, :].shape)


transpose: (4, 3)
ravel: (12,)
flatten: (12,)
一维转置: (3,)
列向量: (3, 1)
行向量: (1, 3)


## 4. `axis`：最容易混淆、也最重要

对形状 `(3, 4)` 的数组：

- `axis=0`：压缩第 0 维（行），结果按列保留，形状为 `(4,)`；
- `axis=1`：压缩第 1 维（列），结果按行保留，形状为 `(3,)`；
- `keepdims=True`：保留被压缩的维度，便于后续广播。

可记成：**axis 指定“要消失的维度”**。


In [9]:
scores = np.array([[80, 90, 70],
                   [60, 75, 85],
                   [95, 88, 92]])

# axis=0 消除“行”维度，得到每列统计量；axis=1 得到每行统计量。
subject_mean = scores.mean(axis=0)                 # 每列平均
student_mean = scores.mean(axis=1)                 # 每行平均
# keepdims=True 保留 (行数, 1)，便于广播回原矩阵。
row_center = scores - scores.mean(axis=1, keepdims=True)

print("每科平均:", subject_mean, subject_mean.shape)
print("每人平均:", student_mean, student_mean.shape)
print("每行中心化:\n", row_center)


每科平均: [78.333 84.333 82.333] (3,)
每人平均: [80.    73.333 91.667] (3,)
每行中心化:
 [[  0.     10.    -10.   ]
 [-13.333   1.667  11.667]
 [  3.333  -3.667   0.333]]


## 5. 广播：从右向左比较形状

两个维度兼容的条件：它们相等，或其中一个为 `1`。缺失的左侧维度视为 `1`。

例：`(3, 4) + (4,)` → 把 `(4,)` 看成 `(1, 4)`，复制到 3 行。  
`(3, 4) + (3,)` **不兼容**；若想给每行加一个数，应把它改为 `(3, 1)`。


In [10]:
matrix = np.arange(12).reshape(3, 4)
column_bias = np.array([10, 20, 30, 40])  # shape (4,)
row_bias = np.array([100, 200, 300]).reshape(3, 1)

print("每列加不同值:\n", matrix + column_bias)
print("每行加不同值:\n", matrix + row_bias)


每列加不同值:
 [[10 21 32 43]
 [14 25 36 47]
 [18 29 40 51]]
每行加不同值:
 [[100 101 102 103]
 [204 205 206 207]
 [308 309 310 311]]


## 6. 元素运算与矩阵运算

`*` 是逐元素乘法，`@` 是矩阵乘法。机器学习中 `X @ w` 十分常见。


In [11]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("逐元素乘法:\n", a * b)
print("矩阵乘法:\n", a @ b)

X = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])          # 3 个样本，2 个特征
w = np.array([0.5, -1.0])           # 每个特征一个权重
predictions = X @ w
print("X @ w:", predictions, "shape:", predictions.shape)


逐元素乘法:
 [[ 5 12]
 [21 32]]
矩阵乘法:
 [[19 22]
 [43 50]]
X @ w: [-1.5 -2.5 -3.5] shape: (3,)


### Universal Functions（ufunc）

`np.sin`、`np.exp`、`np.sqrt` 等函数会逐元素计算。`np.dot` 对一维向量是点积；二维矩阵建议使用更直观的 `@`。


In [12]:
angles = np.array([0, np.pi / 2, np.pi])
print("sin:", np.sin(angles))

left = np.array([1, 2, 3])
right = np.array([4, 5, 6])
print("np.dot:", np.dot(left, right))
print("@:", left @ right)


sin: [0. 1. 0.]
np.dot: 32
@: 32


## 7. 聚合、排序与缺失值


In [13]:
data = np.array([3.0, np.nan, 1.0, 5.0])
print("普通 mean:", data.mean())          # 只要有 nan，结果就是 nan
print("忽略 nan:", np.nanmean(data))
print("是否有限:", np.isfinite(data))

clean = data[np.isfinite(data)]
print("排序:", np.sort(clean))
print("最大值在 clean 中的位置:", np.argmax(clean))


普通 mean: nan
忽略 nan: 3.0
是否有限: [ True False  True  True]
排序: [1. 3. 5.]
最大值在 clean 中的位置: 2


### 常用统计、位置与累计运算

- `min/max/mean/median`：统计量；
- `argmin/argmax`：最小/最大值的位置；
- `cumsum`：累计和；
- `diff`：相邻元素之差；
- `nonzero`：非零元素索引；
- `clip`：把数值限制到指定范围。

这些函数大多支持 `axis`，含义仍是“压缩哪个维度”。


In [14]:
values = np.array([3, 1, 4, 1, 5, 9])

print("min/max:", np.min(values), np.max(values))
print("mean/median:", np.mean(values), np.median(values))
print("argmin/argmax:", np.argmin(values), np.argmax(values))
print("cumsum:", np.cumsum(values))
print("diff:", np.diff(values))
print("nonzero:", np.nonzero(values > 3)[0])
print("clip 到 [2, 5]:", np.clip(values, 2, 5))


min/max: 1 9
mean/median: 3.8333333333333335 3.5
argmin/argmax: 1 5
cumsum: [ 3  4  8  9 14 23]
diff: [-2  3 -3  4  4]
nonzero: [2 4 5]
clip 到 [2, 5]: [3 2 4 2 5 5]


## 8. 合并与拆分

- `concatenate`：沿已有轴连接；
- `stack`：创建一个新轴；
- `vstack` / `hstack`：常用的纵向 / 横向快捷方式。


In [ ]:
import numpy as np
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6]])
# concatenate 沿已有 axis 连接；stack 会创建一个新的 axis。
vertical = np.concatenate([a, b], axis=0)
new_axis = np.stack([a, a], axis=0)
print("vertical:", vertical)
print("new_axis:",new_axis)

print("concatenate shape:", vertical.shape)
print("stack shape:", new_axis.shape)
print("按行拆成 3 份:", np.split(vertical, 3, axis=0))


vertical: [[1 2]
 [3 4]
 [5 6]]
new_axis: [[[1 2]
  [3 4]]

 [[1 2]
  [3 4]]]
concatenate shape: (3, 2)
stack shape: (2, 2, 2)
按行拆成 3 份: [array([[1, 2]]), array([[3, 4]]), array([[5, 6]])]


### `vstack`、`hstack` 与多种拆分

- `vstack`：纵向堆叠，通常增加行；
- `hstack`：横向拼接，二维时通常增加列；
- `split`：要求能够平均拆分；
- `array_split`：允许不能整除的拆分；
- `vsplit` / `hsplit`：按行 / 列拆分二维数组。

合并前先检查除拼接轴外的其他维度是否一致。


In [16]:
top = np.array([[1, 2], [3, 4]])
bottom = np.array([[5, 6]])

print("vstack:\n", np.vstack([top, bottom]))
print("hstack:\n", np.hstack([top, top]))

sequence = np.arange(10)
print("array_split 3 份:", np.array_split(sequence, 3))
print("按列拆分:", np.hsplit(np.arange(12).reshape(3, 4), 2))


vstack:
 [[1 2]
 [3 4]
 [5 6]]
hstack:
 [[1 2 1 2]
 [3 4 3 4]]
array_split 3 份: [array([0, 1, 2, 3]), array([4, 5, 6]), array([7, 8, 9])]
按列拆分: [array([[0, 1],
       [4, 5],
       [8, 9]]), array([[ 2,  3],
       [ 6,  7],
       [10, 11]])]


## 9. 向量化：避免逐元素 Python 循环

向量化代码通常更短，也能调用底层优化实现。先确认形状，再写数组表达式。


In [17]:
values = np.arange(1_000_000, dtype=float)

# 向量化：一次描述全部元素
vectorized = values ** 2 + 2 * values + 1
print(vectorized[:5])

# 条件选择：where(条件, 条件为真时的值, 条件为假时的值)
# where 类似逐元素 if：条件为真选第二项，否则选第三项。
labels = np.where(values[:10] % 2 == 0, "even", "odd")
print(labels)


[ 1.  4.  9. 16. 25.]
['even' 'odd' 'even' 'odd' 'even' 'odd' 'even' 'odd' 'even' 'odd']


## 10. 线性代数与可复现随机数


In [18]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
b = np.array([5.0, 7.0])

# solve 直接求解 Ax=b，比显式计算 inv(A) @ b 更稳定。
solution = np.linalg.solve(A, b)
print("Ax=b 的解:", solution)
print("验证 A@x:", A @ solution)

rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(7)
print("相同种子得到相同序列:", np.array_equal(rng_a.integers(10, size=5),
                                            rng_b.integers(10, size=5)))


Ax=b 的解: [1.6 1.8]
验证 A@x: [5. 7.]
相同种子得到相同序列: True


## 11. 小项目：按列标准化特征

标准化公式：`z = (x - mean) / std`。均值和标准差按特征列计算，并保留二维形状以便广播。


In [19]:
X = np.array([[170.0, 65.0],
              [180.0, 80.0],
              [160.0, 55.0]])  # 列分别是身高、体重

mean = X.mean(axis=0, keepdims=True)
std = X.std(axis=0, keepdims=True)
X_scaled = (X - mean) / std

print("mean shape:", mean.shape)
print("标准化结果:\n", X_scaled)
print("检查列均值:", X_scaled.mean(axis=0))
print("检查列标准差:", X_scaled.std(axis=0))


mean shape: (1, 2)
标准化结果:
 [[ 0.    -0.162]
 [ 1.225  1.298]
 [-1.225 -1.136]]
检查列均值: [ 0. -0.]
检查列标准差: [1. 1.]


## 12. 常见坑与练习

- `np.empty` 是未初始化内存，不代表全 0。
- `arr.T` 对一维数组 `(n,)` 形状不变；列向量应使用 `arr[:, None]`。
- 比较浮点数用 `np.isclose` / `np.allclose`，不要直接依赖 `==`。
- 切片可能是视图；需要隔离修改时 `.copy()`。
- 报形状错误时，第一步打印每个数组的 `.shape`。

**练习**

1. 对一个 `(5, 3)` 数组按列做 min-max 缩放；
2. 计算两组二维点之间的欧氏距离；
3. 将成绩矩阵中低于 60 的元素替换为 60。


In [20]:
# 参考答案
sample = rng.normal(size=(5, 3))
col_min = sample.min(axis=0, keepdims=True)
col_max = sample.max(axis=0, keepdims=True)
scaled = (sample - col_min) / (col_max - col_min)

points_a = np.array([[0, 0], [1, 1]])
points_b = np.array([[3, 4], [4, 5]])
# norm(..., axis=1) 对每一行计算欧氏长度，因此得到每对点的距离。
distances = np.linalg.norm(points_a - points_b, axis=1)

grades = np.array([[58, 90], [76, 49]])
adjusted = np.where(grades < 60, 60, grades)

print("min-max:\n", scaled)
print("距离:", distances)
print("调整后成绩:\n", adjusted)


min-max:
 [[0.568 0.    0.542]
 [0.004 0.828 1.   ]
 [0.532 1.    0.821]
 [0.    0.475 0.   ]
 [1.    0.184 0.446]]
距离: [5. 5.]
调整后成绩:
 [[60 90]
 [76 60]]
